# One hot encoding
In this notebook I will explore the One-Hot encoding method for every non-numerical feature. More specifically,

`"age"`: stays as it is

`"workclass"`: One-Hot encoding

`"fnlwgt"`: drop

`"education"`: One-Hot encoding

`"education-num"`: drop as `education` is represented by One-Hot encoding

`"marital-status"`: One-Hot encoding

`"occupation"`: One-Hot encoding

`"relationship"`: One-Hot encoding

`"race"`: One-Hot encoding

`"sex"`: One-Hot encoding

`"capital-gain"`: stays as it is

`"capital-loss"`: stays as it is

`"hours-per-week"`: stays as it is

`"native-country"`: One-Hot encoding

Here: Unknown data, ie rows with `?` are simply dropped.

For future reference: 
`education`: acceptable for a one-hot experiment, but conceptually this is also a strong candidate for ordinal encoding because it has a meaningful order.

`native-country`: acceptable to one-hot as a baseline, but it is high-cardinality, so it is also a good feature to later compare against frequency or target encoding.

NOTE: Of course the methods introduced here can be apllied differently, fe. we can choose if `"sex"` is One-Hot encoded or not

Here is the raw data loaded and apply
$$
y =
\begin{cases}
1 & \text{if income } > \$50K \\
0 & \text{otherwise}
\end{cases}
$$
to the data set. Also drop the columns which were stated before 

In [1]:
# Enable autoreloading of imported modules
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import pandas as pd
from courselib.utils.splits import train_test_split

# Add the repo root (two levels up from this notebook) to sys.path
sys.path.insert(0, os.path.abspath("../../"))

In [2]:
file_name = "adult.data"
column_names = column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "Class"
]
df = pd.read_csv(file_name, names= column_names,skipinitialspace=True)

In [3]:
target_variable_map = {
    "<=50K": 0,
    ">50K": 1,
}

columns_to_be_dropped = [
    "fnlwgt",
    "education-num"
]

In [4]:
df= df.drop(columns=columns_to_be_dropped)
df["Class"] = df["Class"].map(target_variable_map)
df

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,Class
0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
32557,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
32558,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
32559,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


## One-Hot-Encoding method

In [5]:
def One_Hot_Encoder(df,column_to_be_encoded,number_if_True = 1, number_if_False = 0, delete_old_column = False, drop_unknown_data_rows = [] ):
    """takes a pandas dataframe and columns which need to be encoded and creates new column according to the columns categories.
       The rows then get marked wether the row has or has not the column property. 
       outputs then a new edited copy of df.

    Args:
        df (pd.dataframe): just the dataset
        column_to_be_encoded (list): a list of column names form df which need to be encoded
        number_if_True (int, optional): number if the column property applies to the row. Defaults to 1.
        number_if_False (int, optional): number if the column property does not apply to the row. Defaults to 0.
        delete_old_column (bool, optional): Wether the columns form column_to_be_encoded are deleted or not. Defaults to False.
        drop_unknown_data_rows (list, optional): Drops all rows which include the strings which are in this list. Defaults to empty list.
    """
    _df = df.copy()

    row_amount_before = len(_df)

    if drop_unknown_data_rows !=[]:
        for column_name in column_to_be_encoded:
            _df = _df[~_df[column_name].isin(drop_unknown_data_rows)]

    amount_of_rows_deleted = row_amount_before - len(_df)

    _df = _df.reset_index(drop=True)

    row_amount = len(_df)
    
    for i,column_name in enumerate(column_to_be_encoded):
        list_of_categories = [] #for each column we have a list of categories
        
        for j in range(row_amount):
            
            category =_df[column_name][j] #what string does this row have in the column we look at
            
            if category in list_of_categories:
                new_column_name = column_name +"_"+ category
                _df.loc[j, new_column_name] = number_if_True
                
            if category not in list_of_categories:
                
                list_of_categories.append(category)

                new_column_name = column_name +"_"+ category
                _df[new_column_name] = number_if_False  #add new column
                
                _df.loc[j, new_column_name] = number_if_True
                
            if j== row_amount-1: #delete old column if neccessary
                if delete_old_column:
                    _df = _df.drop(columns=[column_name])
    
    #print(f"{amount_of_rows_deleted} out of {row_amount_before} were deleted, ie.{(1- amount_of_rows_deleted/row_amount_before)*100}% still remain ")            
    return _df

## Method for drpping unknow data

In [6]:
def drop_unknown_rows(df, drop_unknown_data_rows):
    """Drops all rows which include one of the values in drop_unknown_data_rows in any column.
       outputs then a new edited copy of df.

    Args:
        df (pd.dataframe): just the dataset
        drop_unknown_data_rows (list): Drops all rows which include the strings which are in this list.
    """
    _df = df.copy()

    row_amount_before = len(_df)

    for column_name in _df.columns:
        _df = _df[~_df[column_name].isin(drop_unknown_data_rows)]

    amount_of_rows_deleted = row_amount_before - len(_df)

    _df = _df.reset_index(drop=True)

    print(f"{amount_of_rows_deleted} out of {row_amount_before} were deleted, ie.{(1- amount_of_rows_deleted/row_amount_before)*100}% still remain ")

    return _df

## Formatting the Data Set

### Choose the Columns which need to be encoded

In [ ]:
one_hot_cols = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]


### Running the Data though the One-Hot-Encoder

In [11]:
df_encoded = drop_unknown_rows(df,["?"])
df_encoded = One_Hot_Encoder(df,one_hot_cols,delete_old_column=True)

2399 out of 32561 were deleted, ie.92.63229016307852% still remain 


### Test if the Encoded data Runs through the `train_test_split`

In [12]:
X, Y, X_train, Y_train, X_test, Y_test =  train_test_split(df_encoded, 
                                                           training_data_fraction=0.8, 
                                                           return_numpy=True)

print('Training data split as follows:')
print(f'  Training data samples: {len(X_train)}')
print(f'      Test data samples: {len(X_test)}')

Training data split as follows:
  Training data samples: 26049
      Test data samples: 6512


In [13]:
df_encoded

,age,education,capital-gain,capital-loss,hours-per-week,Class,workclass_State-gov,workclass_Self-emp-not-inc,workclass_Private,workclass_Federal-gov,...,native-country_Outlying-US(Guam-USVI-etc),native-country_Scotland,native-country_Trinadad&Tobago,native-country_Greece,native-country_Nicaragua,native-country_Vietnam,native-country_Hong,native-country_Ireland,native-country_Hungary,native-country_Holand-Netherlands
0,39,Bachelors,2174,0,40,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,50,Bachelors,0,0,13,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,38,HS-grad,0,0,40,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,53,11th,0,0,40,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,28,Bachelors,0,0,40,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Assoc-acdm,0,0,38,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
32557,40,HS-grad,0,0,40,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
32558,58,HS-grad,0,0,40,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
32559,22,HS-grad,0,0,20,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
